In [1]:
from sqlalchemy import create_engine
import pandas as pd
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()
database_url = os.getenv('DATABASE_URL')
engine = create_engine(database_url)
print("db 연결 성공!")

db 연결 성공!


In [3]:
query = '''
SELECT nct_id, agency_class, lead_or_collaborator
FROM ctgov.sponsors;
'''
df_sponsors = pd.read_sql(query, engine)
df_sponsors.head()

,nct_id,agency_class,lead_or_collaborator
0,NCT02225418,OTHER,collaborator
1,NCT05122624,NIH,collaborator
2,NCT03981146,INDUSTRY,collaborator
3,NCT02966132,OTHER,collaborator
4,NCT02778035,OTHER,collaborator


In [4]:
df_sponsors.info()

<class 'pandas.DataFrame'>
RangeIndex: 919137 entries, 0 to 919136
Data columns (total 3 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   nct_id                919137 non-null  str  
 1   agency_class          918173 non-null  str  
 2   lead_or_collaborator  919137 non-null  str  
dtypes: str(3)
memory usage: 21.0 MB


In [5]:
df_sponsors['agency_class'].value_counts()

agency_class
OTHER        589763
INDUSTRY     180966
UNKNOWN       49810
NIH           46701
OTHER_GOV     31873
FED           10776
NETWORK        7435
INDIV           761
AMBIG            88
Name: count, dtype: int64

- other(기타): 대학, 병원, NGO 등. 학술적 목적의 연구가 가장 많음
- industry(산업계): 제약회사나 의료기기 업체. 신약승인을 목적으로 하는 상업적 임상시험이 주 
- unknown: 정보 기재되지 않음
- nih(미국 국립보건원): 공익적 연구나 기초의학연구 후원
- other_gov(기타 정부 기관): NIH 제외한 미국 지자체 혹은 기타 정부부처에서 지원하는 연구
- fed(연방정부): 보훈부(VA) 국방부(DoD) 등
- network: 여러 병원이나 대학이 연합하여 대규모 연구 그룹
- indiv: 특정 연구자 개인이 후원
- ambig(모호함): 기관의 성격의 분류 모호한 경우

In [6]:
df_sponsors['lead_or_collaborator'].value_counts()

lead_or_collaborator
lead            575074
collaborator    344063
Name: count, dtype: int64

In [7]:
# 전체 데이터 중 중복된 행의 갯수 확인

total_rows = len(df_sponsors)
duplicate_rows = df_sponsors.duplicated().sum()

print(f"전체 데이터 개수: {total_rows}")
print(f"중복된 행 개수: {duplicate_rows}")
print(f"중복 비율: {(duplicate_rows / total_rows) * 100:.2f}%")

전체 데이터 개수: 919137
중복된 행 개수: 118981
중복 비율: 12.94%


In [9]:
df_duplicates = df_sponsors[df_sponsors.duplicated(keep=False)].sort_values(by='nct_id')
print("--- 중복 데이터 샘플 (최대 10개) ---")
display(df_duplicates.head(10))

--- 중복 데이터 샘플 (최대 10개) ---


,nct_id,agency_class,lead_or_collaborator
164073,NCT00000134,OTHER,collaborator
164072,NCT00000134,OTHER,collaborator
164063,NCT00000134,NIH,collaborator
164065,NCT00000134,OTHER,collaborator
164066,NCT00000134,OTHER,collaborator
164067,NCT00000134,OTHER,collaborator
164068,NCT00000134,OTHER,collaborator
164069,NCT00000134,OTHER,collaborator
164064,NCT00000134,NIH,collaborator
164077,NCT00000134,OTHER,collaborator


In [11]:
# 중복 제거
df_sponsors = df_sponsors.drop_duplicates()
print(df_sponsors.shape)

(800156, 3)


In [14]:
# 스폰서 타입별 카테고리 범주화하기
def sponsor_type(val):
    if val == 'INDUSTRY':
        return 'Industry'
    elif val in ['NIH', 'FED', 'OTHER_GOV']:
        return 'Goverment'
    elif val in ['OTHER', 'NETWORK', 'INDIV', 'AMBIG']:
        return 'Other'
    else:
        return 'Unknown'

df_sponsors['agency_class'] = df_sponsors['agency_class'].apply(sponsor_type)
df_sponsors['agency_class'].value_counts()

agency_class
Other        510099
Industry     175343
Goverment     83809
Unknown       30905
Name: count, dtype: int64

In [15]:
# 원핫인코딩
df_sponsors = pd.get_dummies(df_sponsors, columns=['agency_class', 'lead_or_collaborator'], prefix=['sponsor', 'role'], dtype=int)
# 'nct_id' 기준으로 그룹화
df_sponsors = df_sponsors.groupby('nct_id').max().reset_index()
df_sponsors.info()


<class 'pandas.DataFrame'>
RangeIndex: 575074 entries, 0 to 575073
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   nct_id             575074 non-null  str  
 1   sponsor_Goverment  575074 non-null  int64
 2   sponsor_Industry   575074 non-null  int64
 3   sponsor_Other      575074 non-null  int64
 4   sponsor_Unknown    575074 non-null  int64
 5   role_collaborator  575074 non-null  int64
 6   role_lead          575074 non-null  int64
dtypes: int64(6), str(1)
memory usage: 30.7 MB


In [16]:
# 최종데이터에서 중복값 확인
assert df_sponsors['nct_id'].is_unique, 'Duplicate trial entries found!'

In [17]:
# 최종 데이터셋 저장
target_path = r'C:\dev\clinicaltrials-study\data\processed'
file_name = 'sponsor_clean.csv'
full_save_path = os.path.join(target_path, file_name)
df_sponsors.to_csv(full_save_path, index=False)